# 01 · Setup & Core Library

**Project:** Carbon-Adaptive Reward Shaping for Smart Grid Battery Dispatch (CityLearn + PPO)

This notebook only **defines** things — it does not train or evaluate anything. Every other notebook
loads it with `%run ./01_setup_and_core.ipynb`, so the config, reward functions, environment wrapper and
train/evaluate helpers are written exactly once.

| Section | Contents | Original file |
|---|---|---|
| 1 | Imports | — |
| 2 | Configuration (dataset, seeds, observation indices, PPO hyper-parameters, paths) | `code/config.py` |
| 3 | Reward functions (cost-only, static carbon, dynamic carbon) | `code/rewards.py` |
| 4 | CityLearn → Gymnasium / Stable-Baselines3 wrapper | `code/env_utils.py` |
| 5 | Training callback | `code/train_baselines.py`, `code/train_carbon_agents.py` (was duplicated) |
| 6 | Episode runners & metrics (RBC, PPO policy, self-consumption) | `code/train_*.py`, `code/evaluate.py` (eval loop was copied 3×) |
| 7 | Train-and-evaluate helper for a PPO agent | `code/train_baselines.py`, `code/train_carbon_agents.py` |

### Run order
1. `01_setup_and_core.ipynb` — definitions only (run automatically by the others)
2. `02_sanity_checks.ipynb` — environment verification + reward unit tests
3. `03_training.ipynb` — RBC, Cost-Only PPO, static-λ sweep, Static PPO, Dynamic PPO (**slow**)
4. `04_evaluation_and_plots.ipynb` — re-evaluates every checkpoint, summary table, Pareto frontier, training curves
5. `05_policy_explainability.ipynb` — how each agent uses the battery under different price / carbon conditions

## 1. Imports
`torch` must be imported **before** pandas/CityLearn on Windows, otherwise loading the DLLs fails with `WinError 1114`.

In [ ]:
# Import torch FIRST to prevent DLL conflict (WinError 1114) with pandas on Windows
import torch
import os
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import gymnasium as gym
from citylearn.citylearn import CityLearnEnv
from citylearn.agents.rbc import BasicRBC
from stable_baselines3 import PPO
from stable_baselines3.common.callbacks import BaseCallback

warnings.filterwarnings('ignore')

## 2. Configuration
Single source of truth for the dataset, seeds, observation indices, PPO hyper-parameters and output paths.

- **Dataset:** `citylearn_challenge_2022_phase_1` — 5 residential buildings, each with rooftop PV and a battery (BESS).
  One episode = 8,760 hourly steps (one year).
- **Observation indices** point at features inside the central observation vector (building 0's block).
- **PPO:** standard SB3 settings for continuous-control dispatch; 50,000 training steps ≈ 5.7 episodes.
- **Paths:** the notebooks live in `01_notebook_pipeline/notebooks/`, and results are written to `01_notebook_pipeline/results/`.

In [ ]:
# Notebooks live in <repo>/01_notebook_pipeline/notebooks; results live in <repo>/01_notebook_pipeline/results
# PIPELINE_DIR is found by name, so this works whether Jupyter starts in the repo root, the pipeline folder or notebooks/
PIPELINE_DIR_NAME = '01_notebook_pipeline'
_cwd = Path.cwd().resolve()
PIPELINE_DIR = next((p for p in [_cwd, *_cwd.parents] if p.name == PIPELINE_DIR_NAME), _cwd / PIPELINE_DIR_NAME)
REPO_ROOT = PIPELINE_DIR.parent

# Dataset & environment setup
DATASET_NAME = 'citylearn_challenge_2022_phase_1'
NUM_BUILDINGS = 5
SEEDS = [42, 100, 2026]

# Indices for observation variables (based on building 0 observation list)
CARBON_INTENSITY_IDX = 19
NON_SHIFTABLE_LOAD_IDX = 20
SOLAR_GENERATION_IDX = 21
SOC_IDX = 22
NET_ELECTRICITY_CONSUMPTION_IDX = 23
PRICING_IDX = 24

# PPO training hyperparameters (standard SB3 settings for energy dispatch tasks)
PPO_PARAMS = {
    'learning_rate': 3e-4,
    'n_steps': 2048,
    'batch_size': 128,
    'n_epochs': 10,
    'gamma': 0.99,
    'gae_lambda': 0.95,
    'clip_range': 0.2,
    'ent_coef': 0.0,
    'verbose': 0,
}

# Neural network architecture (separate actor / critic MLPs)
POLICY_KWARGS = {
    'net_arch': {
        'pi': [64, 64],
        'vf': [64, 64]
    }
}

# Training steps (about 5.7 episodes since 1 episode is 8760 steps)
TOTAL_TIMESTEPS = 50_000

# Carbon weight used by the Static and Dynamic agents in the main comparison (chosen from the sweep)
LAMBDA_MAIN = 1.0

# Static weight with the SAME yearly average as the dynamic agent's weight: LAMBDA_MAIN * mean(I_hat) = 1.0 * 0.407.
# The "Static-Matched" baseline separates "weight follows carbon intensity" from "weight is smaller on average".
LAMBDA_MATCHED = 0.41

# Output directories
RESULTS_DIR = PIPELINE_DIR / 'results'
PLOTS_DIR = RESULTS_DIR / 'plots'
TABLES_DIR = RESULTS_DIR / 'tables'
MODELS_DIR = RESULTS_DIR / 'models'
for d in [PLOTS_DIR, TABLES_DIR, MODELS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# Method display names -> reward_type keys and carbon weights
METHODS = ['RBC', 'Cost-Only', 'Static', 'Static-Matched', 'Dynamic']
PPO_METHODS = [m for m in METHODS if m != 'RBC']
REWARD_TYPE_OF = {'Cost-Only': 'cost_only', 'Static': 'static', 'Static-Matched': 'static', 'Dynamic': 'dynamic'}
LAMBDA_OF = {'Cost-Only': None, 'Static': LAMBDA_MAIN, 'Static-Matched': LAMBDA_MATCHED, 'Dynamic': LAMBDA_MAIN}
METHOD_COLORS = {
    'RBC': '#7f8c8d',             # gray
    'Cost-Only': '#e74c3c',       # red
    'Static': '#f39c12',          # orange
    'Static-Matched': '#8e44ad',  # purple
    'Dynamic': '#2ecc71',         # green
}

def run_name(method, seed):
    """File-name stem of a PPO method's model and tables. Static-Matched includes lambda to stay distinct from Static."""
    if method == 'Static-Matched':
        return f"ppo_static_lambda_{LAMBDA_MATCHED}_seed_{seed}"
    return f"ppo_{REWARD_TYPE_OF[method]}_seed_{seed}"

print(f"Configuration loaded. Results folder: {RESULTS_DIR}")

## 3. Reward functions
All rewards are computed from the step that **just finished**.

| Agent | Reward $R_t$ |
|---|---|
| Cost-Only | $-\text{cost}_t$ |
| Static carbon-aware | $-\text{cost}_t - \lambda \cdot \text{carbon}_t$ |
| Dynamic carbon-adaptive (proposed) | $-\text{cost}_t - \lambda(I_t) \cdot \text{carbon}_t$, with $\lambda(I_t) = \lambda_0 \cdot \hat I_t$ |

where $\hat I_t = \text{clip}\left(\frac{I_t - I_{\min}}{I_{\max} - I_{\min}}, 0, 1\right)$ is the grid's marginal carbon intensity
normalised over the dataset. When the grid is cleanest the dynamic agent behaves like Cost-Only ($\lambda = 0$);
when it is dirtiest it behaves like Static with $\lambda = \lambda_0$.

**Note on indexing:** CityLearn's building arrays (`net_electricity_consumption_cost`, `..._emission`) are sliced up to
`time_step + 1`, so after `env.step()` the last element `[-1]` is the cost/emission caused by the action just taken
(verified empirically in notebook 02).

In [ ]:
def get_step_metrics(env):
    """
    Extracts the operational cost and carbon emissions of the step that was just completed,
    summed across all buildings (last element of each building's history array).
    """
    cost = sum(float(b.net_electricity_consumption_cost[-1]) for b in env.buildings)
    carbon = sum(float(b.net_electricity_consumption_emission[-1]) for b in env.buildings)
    return cost, carbon

def get_carbon_intensity(env):
    """
    Marginal carbon intensity of the step that was just completed.
    env.time_step is incremented by env.step(), so the completed step is time_step - 1.
    """
    step_idx = max(0, env.time_step - 1)
    return float(env.buildings[0].carbon_intensity._carbon_intensity[step_idx])

# 1. Cost-only reward: R = -cost
def cost_only_reward(env, action):
    cost, _ = get_step_metrics(env)
    return -cost

# 2. Static-weighted carbon-aware reward: R = -cost - lambda * carbon
def static_carbon_reward(env, action, lambda_val):
    cost, carbon = get_step_metrics(env)
    return -cost - (lambda_val * carbon)

# 3. Dynamic carbon-adaptive reward: R = -cost - lambda(intensity) * carbon
def dynamic_carbon_reward(env, action, lambda_base, carbon_min, carbon_max):
    cost, carbon = get_step_metrics(env)
    intensity = get_carbon_intensity(env)

    # Normalised intensity in [0, 1]
    if carbon_max > carbon_min:
        norm_intensity = (intensity - carbon_min) / (carbon_max - carbon_min)
    else:
        norm_intensity = 0.0

    # Clip to [0, 1] to guard against out-of-bounds testing data
    norm_intensity = np.clip(norm_intensity, 0.0, 1.0)

    # Linear scaling phi(I) = I_hat: lambda ranges from 0 (cleanest grid) to lambda_base (dirtiest grid)
    lambda_dynamic = lambda_base * norm_intensity
    return -cost - (lambda_dynamic * carbon)

## 4. Environment wrapper
CityLearn 2.1.2 still uses the old `gym` API, while Stable-Baselines3 2.x expects `gymnasium`. The wrapper:

- converts `reset()` → `(obs, info)` and `step()` → 5-tuple `(obs, reward, terminated, truncated, info)`;
- exposes the **centralised** observation/action space (one agent controls all 5 batteries → action shape `(5,)`);
- replaces CityLearn's default reward with one of the custom rewards above;
- precomputes $I_{\min}, I_{\max}$ from the dataset for the dynamic reward;
- puts per-step and per-episode cost / carbon / reward into `info` for logging.

In [ ]:
class CityLearnStableBaselinesWrapper(gym.Env):
    """
    Gymnasium wrapper that adapts CityLearnEnv to work with Stable-Baselines3 and applies a custom reward
    ('default', 'cost_only', 'static' or 'dynamic').
    """
    def __init__(self, env, reward_type='default', lambda_val=None, verbose=False):
        super().__init__()
        self.env = env
        self.reward_type = reward_type
        self.lambda_val = lambda_val

        # Centralised spaces: convert old gym Box spaces to gymnasium Box spaces (SB3 type assertions)
        old_obs_space = env.observation_space[0]
        old_act_space = env.action_space[0]
        self.observation_space = gym.spaces.Box(
            low=old_obs_space.low, high=old_obs_space.high, dtype=old_obs_space.dtype
        )
        self.action_space = gym.spaces.Box(
            low=old_act_space.low, high=old_act_space.high, dtype=old_act_space.dtype
        )

        # Precompute carbon min/max from dataset for dynamic reward normalisation
        carbon_intensities = self.env.buildings[0].carbon_intensity._carbon_intensity
        self.carbon_min = float(np.min(carbon_intensities))
        self.carbon_max = float(np.max(carbon_intensities))

        self.episode_cost = 0.0
        self.episode_carbon = 0.0
        self.episode_reward = 0.0
        if verbose:
            print(f"Wrapper initialized: reward_type={reward_type}, lambda={lambda_val}")
            print(f"Carbon intensity bounds detected: min={self.carbon_min:.4f}, max={self.carbon_max:.4f}")

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        if seed is not None:
            self.env.random_seed = seed

        self.episode_cost = 0.0
        self.episode_carbon = 0.0
        self.episode_reward = 0.0

        obs = self.env.reset()
        return np.array(obs[0], dtype=np.float32), {}

    def step(self, action):
        # CityLearn expects the central action inside a list: [action]
        obs, default_reward, done, info = self.env.step([action])

        cost, carbon = get_step_metrics(self.env)
        self.episode_cost += cost
        self.episode_carbon += carbon

        if self.reward_type == 'cost_only':
            reward = cost_only_reward(self.env, action)
        elif self.reward_type == 'static':
            assert self.lambda_val is not None, "lambda_val must be specified for static reward"
            reward = static_carbon_reward(self.env, action, self.lambda_val)
        elif self.reward_type == 'dynamic':
            assert self.lambda_val is not None, "lambda_val (base) must be specified for dynamic reward"
            reward = dynamic_carbon_reward(self.env, action, self.lambda_val, self.carbon_min, self.carbon_max)
        else:
            # Default CityLearn reward: sum of building rewards
            reward = float(sum(default_reward))

        self.episode_reward += reward

        info['step_metrics'] = {'cost': cost, 'carbon': carbon, 'reward': reward}
        if done:
            info['episode_metrics'] = {
                'cost': self.episode_cost,
                'carbon': self.episode_carbon,
                'reward': self.episode_reward
            }

        return np.array(obs[0], dtype=np.float32), float(reward), done, False, info

    def render(self):
        return self.env.render()

    def close(self):
        return self.env.close()

def make_env(reward_type='default', lambda_val=None, seed=None, verbose=False):
    """Create a centralised CityLearn environment wrapped for SB3 with the requested reward."""
    env = CityLearnEnv(schema=DATASET_NAME, central_agent=True)
    wrapped_env = CityLearnStableBaselinesWrapper(env, reward_type=reward_type, lambda_val=lambda_val, verbose=verbose)
    if seed is not None:
        wrapped_env.reset(seed=seed)
    return wrapped_env

## 5. Training callback
Records cumulative reward, cost and carbon every time a training episode (one simulated year) finishes.
These logs become the training curves in notebook 04.

In [ ]:
class RLMetricsCallback(BaseCallback):
    """Callback to record training progress (rewards, costs, carbon) on episode completion."""
    def __init__(self, verbose=1):
        super().__init__(verbose)
        self.training_log = []
        self.episode_count = 0

    def _on_step(self) -> bool:
        dones = self.locals.get("dones")
        infos = self.locals.get("infos")

        if dones is not None and infos is not None:
            for idx, done in enumerate(dones):
                if done and 'episode_metrics' in infos[idx]:
                    metrics = infos[idx]['episode_metrics']
                    self.episode_count += 1
                    self.training_log.append({
                        'episode': self.episode_count,
                        'step': self.num_timesteps,
                        'reward': metrics['reward'],
                        'cost': metrics['cost'],
                        'carbon': metrics['carbon']
                    })
                    if self.verbose > 0:
                        print(f"  Episode {self.episode_count} (Step {self.num_timesteps}): "
                              f"Reward={metrics['reward']:.2f}, Cost={metrics['cost']:.2f}, Carbon={metrics['carbon']:.2f}")
        return True

## 6. Episode runners & metrics
One deterministic evaluation episode (a full year) per call.

- `run_rbc_episode(seed)` — CityLearn's `BasicRBC` (decentralised, time-of-day rules).
- `run_policy_episode(model, env, seed)` — a trained PPO policy with `deterministic=True`.
- `compute_self_consumption(buildings)` — share of on-site solar used on-site instead of exported.

Each runner returns a summary dict (`cost`, `carbon`, `self_consumption`) and a per-hour **trace** DataFrame.
The trace stores the carbon intensity / price of the observation the agent **acted on** (the original scripts logged
the *next* observation, which is off by one hour for the explainability analysis).

**Self-consumption fix:** CityLearn stores `solar_generation` as **negative** numbers. The original `evaluate.py`
summed it, got a negative total, and fell into the `else 1.0` branch — so every method reported 100%.
Here we use its absolute value, and only count exports up to the solar produced in that hour (battery discharge
can also cause export).

In [ ]:
def compute_self_consumption(buildings):
    """Fraction of PV generation consumed on-site rather than exported to the grid (index 0 is a placeholder)."""
    total_solar = 0.0
    total_exported_solar = 0.0
    for b in buildings:
        solar = np.abs(np.array(b.solar_generation[1:], dtype=np.float64))
        net_cons = np.array(b.net_electricity_consumption[1:], dtype=np.float64)
        exported = np.maximum(0.0, -net_cons)
        total_solar += solar.sum()
        total_exported_solar += np.minimum(exported, solar).sum()
    return (total_solar - total_exported_solar) / total_solar if total_solar > 0 else float('nan')

def run_rbc_episode(seed):
    """Runs CityLearn's deterministic Rule-Based Controller for one full episode."""
    env = CityLearnEnv(schema=DATASET_NAME, central_agent=False)
    env.random_seed = seed
    obs = env.reset()
    agent = BasicRBC(env)

    step_data = []
    for step in range(env.time_steps):
        actions = agent.predict(obs)
        decision_obs = obs
        obs, reward, done, info = env.step(actions)
        cost, carbon = get_step_metrics(env)

        step_data.append({
            'step': step,
            'cost': cost,
            'carbon': carbon,
            'carbon_intensity': decision_obs[0][CARBON_INTENSITY_IDX],
            'pricing': decision_obs[0][PRICING_IDX],
            'solar_gen': sum(o[SOLAR_GENERATION_IDX] for o in decision_obs),
            'non_shiftable_load': sum(o[NON_SHIFTABLE_LOAD_IDX] for o in decision_obs),
            'avg_soc': sum(o[SOC_IDX] for o in decision_obs) / len(decision_obs),
            'action_b0': actions[0][0]
        })
        if done:
            break

    trace = pd.DataFrame(step_data)
    summary = {
        'method': 'RBC', 'seed': seed,
        'cost': trace['cost'].sum(), 'carbon': trace['carbon'].sum(),
        'self_consumption': compute_self_consumption(env.buildings),
    }
    return summary, trace

def run_policy_episode(model, env, seed, method=''):
    """Runs a trained SB3 policy deterministically for one full episode on a wrapped env."""
    obs, _ = env.reset(seed=seed)

    step_data = []
    for step in range(env.env.time_steps):
        action, _ = model.predict(obs, deterministic=True)
        decision_obs = obs
        obs, reward, terminated, truncated, info = env.step(action)
        metrics = info['step_metrics']

        step_data.append({
            'step': step,
            'cost': metrics['cost'],
            'carbon': metrics['carbon'],
            'reward': metrics['reward'],
            'carbon_intensity': decision_obs[CARBON_INTENSITY_IDX],
            'pricing': decision_obs[PRICING_IDX],
            'soc_b0': decision_obs[SOC_IDX],
            'action_b0': float(action[0])
        })
        if terminated:
            break

    trace = pd.DataFrame(step_data)
    summary = {
        'method': method, 'seed': seed,
        'cost': trace['cost'].sum(), 'carbon': trace['carbon'].sum(),
        'self_consumption': compute_self_consumption(env.env.buildings),
    }
    return summary, trace

## 7. Train-and-evaluate helper
`train_and_evaluate_ppo` is the single function behind Cost-Only, Static, Static-Matched and Dynamic PPO (previously two
near-identical functions in two scripts). For a given reward and seed it:

1. builds the environment and a PPO model with the shared hyper-parameters,
2. trains for `TOTAL_TIMESTEPS` while logging episode metrics,
3. saves the checkpoint to `01_notebook_pipeline/results/models/ppo_<reward>_seed_<seed>.zip`,
4. runs one deterministic evaluation episode and saves `*_train_log.csv`, `*_eval_trace.csv` and `*_summary.csv`
   to `01_notebook_pipeline/results/tables/`.

`model_path()` builds checkpoint names; sweep checkpoints include λ in the name. The `.zip` suffix is explicit,
because a name like `..._lambda_0.2_seed_42` otherwise looks like it already has an extension and SB3 saves it
without one.

In [ ]:
def model_path(reward_type, seed, lambda_val=None):
    """Checkpoint path; sweep runs include lambda in the file name."""
    if lambda_val is None:
        return MODELS_DIR / f'ppo_{reward_type}_seed_{seed}.zip'
    return MODELS_DIR / f'ppo_{reward_type}_lambda_{lambda_val}_seed_{seed}.zip'

def train_ppo(reward_type, seed, lambda_val=None, total_timesteps=TOTAL_TIMESTEPS, verbose=1):
    """Trains a PPO agent with the given reward. Returns (model, env, training_log DataFrame)."""
    env = make_env(reward_type=reward_type, lambda_val=lambda_val, seed=seed)
    model = PPO('MlpPolicy', env, policy_kwargs=POLICY_KWARGS, seed=seed, **PPO_PARAMS)
    callback = RLMetricsCallback(verbose=verbose)
    model.learn(total_timesteps=total_timesteps, callback=callback)
    return model, env, pd.DataFrame(callback.training_log)

def train_and_evaluate_ppo(reward_type, seed, lambda_val=None, method_label=None, name=None):
    """Trains, saves and evaluates one PPO agent; writes train log, eval trace and summary CSVs."""
    name = name or f"ppo_{reward_type}_seed_{seed}"
    print(f"\n=== Training {method_label or reward_type} (seed {seed}, lambda={lambda_val}) ===")

    model, env, df_train = train_ppo(reward_type, seed, lambda_val)
    model.save(MODELS_DIR / f"{name}.zip")
    df_train.to_csv(TABLES_DIR / f"{name}_train_log.csv", index=False)

    summary, trace = run_policy_episode(model, env, seed, method=method_label)
    trace.to_csv(TABLES_DIR / f"{name}_eval_trace.csv", index=False)
    summary['reward'] = df_train['reward'].iloc[-1] if not df_train.empty else 0.0
    pd.DataFrame([summary]).to_csv(TABLES_DIR / f"{name}_summary.csv", index=False)

    print(f"Eval: Cost = {summary['cost']:.2f}, Carbon = {summary['carbon']:.2f}")
    return summary